In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
DATA_PATH = "../data/PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(DATA_PATH)
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 11 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   step            1048575 non-null  int64  
 1   type            1048575 non-null  str    
 2   amount          1048575 non-null  float64
 3   nameOrig        1048575 non-null  str    
 4   oldbalanceOrg   1048575 non-null  float64
 5   newbalanceOrig  1048575 non-null  float64
 6   nameDest        1048575 non-null  str    
 7   oldbalanceDest  1048575 non-null  float64
 8   newbalanceDest  1048575 non-null  float64
 9   isFraud         1048575 non-null  int64  
 10  isFlaggedFraud  1048575 non-null  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 116.4 MB


In [3]:
dtypes = {
    'step': 'int16',            # 1 to 95 fits easily into 16-bit integer (-32,768 to 32,767)
    'type': 'category',         # Only 5 unique values ('TRANSFER', 'CASH_OUT', etc.)
    'amount': 'float32',
    'oldbalanceOrg': 'float32',
    'newbalanceOrig': 'float32',
    'oldbalanceDest': 'float32',
    'newbalanceDest': 'float32',
    'isFraud': 'int8',          # Only 0 or 1
    'isFlaggedFraud': 'int8'
}

df = pd.read_csv(DATA_PATH, dtype=dtypes)
df.info(memory_usage='deep')


<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 11 columns):
 #   Column          Non-Null Count    Dtype   
---  ------          --------------    -----   
 0   step            1048575 non-null  int16   
 1   type            1048575 non-null  category
 2   amount          1048575 non-null  float32 
 3   nameOrig        1048575 non-null  str     
 4   oldbalanceOrg   1048575 non-null  float32 
 5   newbalanceOrig  1048575 non-null  float32 
 6   nameDest        1048575 non-null  str     
 7   oldbalanceDest  1048575 non-null  float32 
 8   newbalanceDest  1048575 non-null  float32 
 9   isFraud         1048575 non-null  int8    
 10  isFlaggedFraud  1048575 non-null  int8    
dtypes: category(1), float32(5), int16(1), int8(2), str(2)
memory usage: 62.0 MB


In [8]:
counts = df['isFraud'].value_counts()
percentages = df['isFraud'].value_counts(normalize = True)*100

summary = pd.DataFrame({
    'Count': counts,
    'Pecentages(%)' : percentages.round(4)
})

summary.index = ['Legitamate(0)', 'Fraud(1)']
print(summary)

                 Count  Pecentages(%)
Legitamate(0)  1047433        99.8911
Fraud(1)          1142         0.1089


In [11]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [14]:
type_summary = df.groupby('type').agg(total_transactions = ('isFraud', 'count'), fraud_cases = ('isFraud', 'sum'), fraud_rate = ('isFraud', lambda x: (x.sum()/ x.count())*100) ).sort_values('fraud_cases', ascending=False)

print(type_summary)

          total_transactions  fraud_cases  fraud_rate
type                                                 
CASH_OUT              373641          578    0.154694
TRANSFER               86753          564    0.650122
CASH_IN               227130            0    0.000000
DEBIT                   7178            0    0.000000
PAYMENT               353873            0    0.000000


In [19]:
# Filter to vulnerable types
vuln_df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

# How often is the account wiped to 0?
emptied = (vuln_df['oldbalanceOrg'] > 0) & (vuln_df['newbalanceOrig'] == 0)

# Check crosstab with isFraud
pd.crosstab(emptied, vuln_df['isFraud'], rownames=['Account Emptied to $0'])


isFraud,0,1
Account Emptied to $0,,
False,267507,29
True,191745,1113


In [29]:
vuln_df['balance_drain_ratio'] = vuln_df['amount'] / (vuln_df['oldbalanceOrg'] + 1.0)
vuln_df.head()

#compare median balance drain ratio with 0 and 1
vuln_df.groupby('isFraud')['balance_drain_ratio'].agg(['mean', 'median', 'max'])

,mean,median,max
isFraud,,,
0,123421.070312,642.772278,5860863.00
1,7484.323242,0.999997,1933920.75


In [34]:
#min,max,total unique hours
print(f"Min step:{df['step'].min()}, Max step:{df['step'].max()}, Total hours: {df['step'].nunique()}")

#total transactions and fraud cases per hour
hourly = df.groupby('step').agg(total_tx = ('isFraud', 'count'), fraud_tx = ('isFraud', 'sum'))
print('\nFirst 5 hours:')
print(hourly.head())

print('\nSummary per hour :')
print(f'Average transactions per hour: {hourly['total_tx'].mean():,.0f}')
print(f'Average frauds per hour: {hourly['fraud_tx'].mean():.2f}')

Min step:1, Max step:95, Total hours: 95

First 5 hours:
      total_tx  fraud_tx
step                    
1         2708        16
2         1014         8
3          552         4
4          565        10
5          665         6

Summary per hour :
Average transactions per hour: 11,038
Average frauds per hour: 12.02
